<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_2_k_scaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_2_k_scaler

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Generación de ventanas X e y para train, valid y test

    En este paso se generan las ventanas de entrada (X) y los targets (y) para los conjuntos de entrenamiento, validación y prueba.
    Se trabaja con un window_size de 90 minutos y se construyen datasets independientes para cada horizonte de predicción: 30, 60 y 90 minutos.

3. Escalado de ventanas

    En este paso se aplica un proceso de normalización/estandarización a las ventanas generadas, utilizando un scaler entrenado únicamente con el set de entrenamiento para cada horizonte de predicción.
    De esta forma, se aseguran valores comparables entre features y se evita data leakage.
    El scaler ajustado se guarda para poder transformar consistentemente los conjuntos de validación y prueba.

4. Guardado de ventanas escaladas

    En este paso se almacenan en disco las ventanas ya escaladas de train, valid y test, correspondientes a cada horizonte de predicción (30, 60 y 90 minutos).
    Esto permite reutilizar los datasets en etapas posteriores sin necesidad de repetir el preprocesamiento.



## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [36]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [37]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [38]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [39]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

In [40]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [41]:
def load_data(fold: str, data: str):

    data_path = f'{drive_path}/5_transformer_90_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar orden cronológico por índice
    df = df.sort_index()

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [42]:
k_folds = [1, 2, 3, 4, 5]

In [43]:
mnq_train = {}
mnq_valid = {}
mnq_test  = {}

for k in k_folds:
    print(f'Cargando datos de Fold {k}..')
    mnq_train[k] = load_data(str(k), 'train')
    mnq_valid[k] = load_data(str(k), 'valid')
    mnq_test[k]  = load_data(str(k), 'test')

Cargando datos de Fold 1..
Cargando datos de Fold 2..
Cargando datos de Fold 3..
Cargando datos de Fold 4..
Cargando datos de Fold 5..


In [44]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      3.00 GB
RAM disponible: 9.36 GB


### 1.2. Información de datasets


In [45]:
def info_dataset(df, name: str):

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  #print(f"\t{name}:\t{num_dias} días")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  #print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  print(f"\t{name}:\t{num_dias} días con {int(promedio_por_fecha)} registros")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo

  #print(f"\tHora diaria de inicio {primer_hora}")
  #print(f"\tHora diaria de final {ultima_hora}")
  #print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [ ]:
#for k in k_folds:
#  print(f'Fold {k}:')
#  info_dataset(globals()[f'mnq_train_{k}'],f'mnq_train_{k}')
#  info_dataset(globals()[f'mnq_valid_{k}'],f'mnq_valid_{k}')
#  info_dataset(globals()[f'mnq_test_{k}'],f'mnq_test_{k}')
#  print ('\n')

### 1.3. Carga de listado de features por ventana de tiempo 2_feature_engineering


In [46]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]

In [47]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


In [ ]:
features_base = ['open', 'high', 'low', 'close', 'volume']
features_90 = features_base + features_to_90

## 2. Generación de ventanas X e y para train, valid y test

Definimos el target de cada horizonte:

In [48]:
target_col_90 = "target_return_90"

Luego definimos el listado de features para cada horizonte:

In [49]:
#features_90 = features_90 = [
#    col for col in mnq_train_1.columns
#    if col not in ["date", "target_return_90"]
#]

In [50]:
features_90 = features_to_90.copy()

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_90`, `rev_mom_z_90` y `ire_90` necesitan 90 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el `window_size` corremos el riesgo de perder información o generar NaNs.

Y un `window_size` más largo?  por ahora experimentemos con 90.


In [51]:
window_size = 90

Definimos las rutas para las ventanas:

In [52]:
# Ruta base donde guardarás los folds
ruta_k_windows = f'{drive_path}/5_transformer_90_model/5_1_k_windows'
os.makedirs(ruta_k_windows, exist_ok=True)

In [53]:
def xy_paths_for_fold(k: int):
    # Crear carpeta del fold
    fold_path = os.path.join(ruta_k_windows, f"fold_{k}")
    os.makedirs(fold_path, exist_ok=True)
    base = f'{drive_path}/5_transformer_90_model/5_1_k_windows/fold_{k}'
    return {
        "X_train": f"{base}/X_train_{k}.npz",
        "y_train": f"{base}/y_train_{k}.npz",
        "X_valid": f"{base}/X_valid_{k}.npz",
        "y_valid": f"{base}/y_valid_{k}.npz",
        "X_test":  f"{base}/X_test_{k}.npz",
        "y_test":  f"{base}/y_test_{k}.npz",
    }

In [54]:
K = 5

rutas_ventanas = {
    k: xy_paths_for_fold(k)
    for k in range(1, K + 1)
}

In [55]:
rutas_ventanas

{1: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_train_1.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_valid_1.npz',
  'y_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_valid_1.npz',
  'X_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_test_1.npz',
  'y_test': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/y_test_1.npz'},
 2: {'X_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_train_2.npz',
  'y_train': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/y_train_2.npz',
  'X_valid': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_2/X_valid_2.npz'

### 2.0. Funciones

#### Función para generar ventanas

Genera ventana consecutivas y no aleatorias, es decir: ventanas deslizantes (sliding windows) dentro de cada día.

- Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.

- Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.

Ejemplo si window_size = 90 y tenemos 301 minutos:

  | Iteración | Ventana usada     | Target extraído       |
  | --------- | ----------------- | --------------------- |
  | i = 0     | registros 0–89    | target = registro 89  |
  | i = 1     | registros 1–90    | target = registro 90  |
  | i = 2     | registros 2–91    | target = registro 91  |
  | ...       | ...               | ...                   |
  | i = 210   | registros 210–299 | target = registro 299 |

Esto da 301 - 90 = 211 ventanas por día, todas consecutivas.


In [56]:
#Función para generar ventanas y vectorizarlas
def generar_ventanas(df, features, target_col, window_size):
    X, y = [], []

    #1. Agrupamiento diario: Cada iteración toma un día completo del dataset (917 días en total). Dentro de ese grupo tenemos 301 registros minuto a minuto.
    for fecha, grupo in tqdm(df.groupby("date"), desc="Procesando días"):
        grupo = grupo.reset_index(drop=True)

        #2. Iteración dentro del día: genera una ventana que empieza en el minuto i  termina en i + windows_size-1.
        for i in range(len(grupo) - window_size):
            ventana = grupo.loc[i:i+window_size-1, features]
            if ventana.isnull().any().any():
                continue

            # 3. Toma las columnas listadas en features (por ejemplo, 10 features por minuto) y las aplanas en un solo vector 1D de longitud window_size × len(features) (= 900 si window_size=90 y len(features)=10).
            vector = ventana.values.flatten()

            #4. El target de la ventana es el valor del registro en el último minuto de la ventana, o sea el del minuto i + window_size - 1.
            #(No es “futuro”, sino el último dentro de la ventana).
            target = grupo.loc[i+window_size-1, target_col]

            X.append(vector)
            y.append(target)

      # Se obtiene:
      # X.shape = (n_ventanas_totales, window_size * n_features)
      # y.shape = (n_ventanas_totales,)
    return np.array(X), np.array(y)

Cada ventana contiene:

- 90 minutos consecutivos de datos de un mismo día.
- En cada minuto, 10 o 12 features (por ejemplo: open, high, low, close, volume, etc.).
- Esos 90×10 (ó 12) valores se aplanan en un vector de 900 (ó 1080) elementos.
- El target asociado es el valor del minuto siguiente al final de la ventana (o del último minuto, según definas).
- Por día se generan 301 − 90 = 211 ventanas, todas superpuestas y consecutivas.
- Repetido en los 917 días de train,  se obtiene 193 487 ventanas en total.



#### Función para generar xy de acuerdo a horizonte de tiempo

In [57]:
def generar_xy (
    df_train,
    df_valid,
    df_test,
    features,
    target: str,
    window_size: int,
    path_xy_train : str,
    path_xy_valid : str,
    path_xy_test : str
    ):

  if not os.path.exists(path_xy_train):
      print(f'El archivo no existe -> Generando X_train e y_train para {target}: ')
      X_train, y_train = generar_ventanas(df_train, features, target, window_size)
      np.savez_compressed(path_xy_train, X=X_train, y=y_train)
      print("Guardado:", path_xy_train)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_train)
      data_train = np.load(path_xy_train)
      X_train, y_train = data_train["X"], data_train["y"]

  if not os.path.exists(path_xy_valid):
      print('El archivo no existe -> Generando X_valid e y_valid: ')
      X_valid, y_valid = generar_ventanas(df_valid, features, target, window_size)
      np.savez_compressed(path_xy_valid, X=X_valid, y=y_valid)
      print("Guardado:", path_xy_valid)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_valid)
      data_valid = np.load(path_xy_valid)
      X_valid, y_valid = data_valid["X"], data_valid["y"]

  if not os.path.exists(path_xy_test):
      print('El archivo no existe -> Generando X_test e y_test: ')
      X_test, y_test = generar_ventanas(df_test, features, target, window_size)
      np.savez_compressed(path_xy_test, X=X_test, y=y_test)
      print("Guardado:", path_xy_test)
  else:
      print("Ya existe -> Cargando desde disco:", path_xy_test)
      data_test = np.load(path_xy_test)
      X_test, y_test = data_test["X"], data_test["y"]

  return X_train, y_train, X_valid, y_valid, X_test, y_test

#### Función para revisar información de ventanas

In [58]:
import psutil
import numpy as np

def xy_info(fold: str, X_train, y_train, X_valid, y_valid, X_test, y_test):

    # RAM total disponible del entorno (Colab o sistema local)
    total_ram_gb = psutil.virtual_memory().total / (1024**3)

    print(40*'-')
    print(f'Información de {fold}:')
    print(40*'-')
    print(f'Memoria total del entorno: {total_ram_gb:.2f} GB')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')

        # Cantidad de muestras
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        # Dimensión del set
        if X.ndim == 2:
            print(f'\tDimensión 2D: {X.shape} (aplanado)')
        elif X.ndim == 3:
            print(f'\tDimensión 3D: {X.shape} (n_samples, window_size, n_features)')

        # Tamaño en memoria
        size_x_gb = X.nbytes / (1024**3)
        size_y_gb = y.nbytes / (1024**3)
        size_total = size_x_gb + size_y_gb
        pct_ram = (size_total / total_ram_gb) * 100

        print(f'\tTamaño X: {size_x_gb:.3f} GB')
        print(f'\tTamaño y: {size_y_gb:.6f} GB')
        print(f'\tTOTAL: {size_total:.3f} GB  →  {pct_ram:.1f}% de la RAM disponible')

        # Estadísticas básicas del target
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, '
              f'min={y.min():.6f}, max={y.max():.6f}')


In [59]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      3.11 GB
RAM disponible: 9.24 GB


## 3. Escalado de ventanas

En este punto se escalan las ventanas de entrada para que todas las features tengan la misma magnitud, entrenando el scaler con los datos de entrenamiento y aplicándolo luego a validación y test.

### 3.0. Funciones


#### 3.0.1. Función para elegir escalador o cargar escalador

In [60]:
def _choose_scaler(scaler_type="standard"):
    st = scaler_type.lower()
    if st in ["standard", "z", "zscore"]:
        return StandardScaler()
    elif st in ["minmax", "min_max"]:
        return MinMaxScaler()
    else:
        raise ValueError("scaler_type debe ser 'standard' o 'minmax'")

#### 3.0.2. Función para entrenar un scaler en datos secuenciales 3D (ventanas), tratándolos como una sola tabla 2D de features.

In [61]:
# -------------------------------------------------------------------------
# _fit_on_3d
#
# Entrada: X_train_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Convierte todas las secuencias en un dataset tabular de features.
# - Ajusta el scaler (ej. StandardScaler) sobre todos los valores de todas
#   las ventanas y pasos, feature por feature.
#
# Resultado: devuelve un scaler entrenado con la estadística global de cada
# feature (media, std, min, max, según el tipo de scaler utilizado).
# -------------------------------------------------------------------------

def _fit_on_3d(X_train_3d, scaler):
    n, W, F = X_train_3d.shape
    scaler.fit(X_train_3d.reshape(-1, F))
    return scaler

#### 3.0.3. Función para aplicar el scaler de _fit_on_3d y devolver los datos escalados, manteniendo la estructura original (n, W, F).

In [62]:
# -------------------------------------------------------------------------
# _transform_3d
#
# Entrada: X_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Aplica la transformación del scaler entrenado (ej. StandardScaler).
# - Restaura la forma original (n, W, F) para conservar la estructura 3D
#   necesaria en modelos secuenciales (RNN, LSTM, Transformers).
#
# Resultado: devuelve el mismo dataset 3D pero con todos los features escalados
# de manera consistente en cada ventana y paso de tiempo.
# -------------------------------------------------------------------------

def _transform_3d(X_3d, scaler):
    n, W, F = X_3d.shape
    Xf = X_3d.reshape(-1, F)
    Xs = scaler.transform(Xf).reshape(n, W, F)
    return Xs

#### 3.0.4. Función para escalar y guardar escalador

In [63]:
import os
import joblib
import numpy as np

def scale_and_save(
    X_train,
    X_valid=None,
    X_test=None,
    scaler_type="standard",
    scaler_path=f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl",
    window_size=None,   # si X_* están en 2D (n_samples, window_size*len(features_{target})), pasá window_size para escalar por feature
    verbose=True,
    reuse_if_exists=True,  # <-- NUEVO: si True y existe scaler_path, lo reutiliza
):
    """
    Escala X_train (y opcionalmente valid/test) y guarda el escalador.
    - Si existe un scaler en `scaler_path` y `reuse_if_exists=True`, lo carga y NO vuelve a hacer fit.
    - Si no existe, crea uno nuevo, hace fit con X_train y lo guarda en `scaler_path`.

    - Si X_* es 3D: (n, W, F) -> fit por feature.
    - Si X_* es 2D: (n, W*F). Si pasás window_size=W, reescala por feature reconstruyendo 3D; si no, escala columnas tal cual.

    Return:
        X_train_scaled, X_valid_scaled (o None), X_test_scaled (o None), scaler
    """
    # ---------------------------------------------------------
    # 0) Asegurar que exista la carpeta del scaler
    # ---------------------------------------------------------
    scaler_dir = os.path.dirname(scaler_path)
    if scaler_dir and not os.path.exists(scaler_dir):
        os.makedirs(scaler_dir, exist_ok=True)

    # ---------------------------------------------------------
    # 1) Obtener scaler: cargar si existe, o crear y ajustar si no
    # ---------------------------------------------------------
    fitted = False

    if reuse_if_exists and os.path.exists(scaler_path):
        # Cargar scaler ya entrenado
        scaler = joblib.load(scaler_path)
        fitted = True
        if verbose:
            print(f"🔁 Usando scaler existente de: {scaler_path}")
    else:
        # Crear scaler nuevo (sin fit todavía)
        scaler = _choose_scaler(scaler_type)
        if verbose:
            if os.path.exists(scaler_path) and not reuse_if_exists:
                print("scaler_path existe pero reuse_if_exists=False → se creará y ajustará un nuevo scaler.")
            else:
                print("Scaler no encontrado → se creará y ajustará uno nuevo.")

    # ---------------------------------------------------------
    # 2) Escalado según dimensión de X_train
    # ---------------------------------------------------------
    if X_train.ndim == 3:
        # X_train: (n, W, F)
        if not fitted:
            scaler = _fit_on_3d(X_train, scaler)

        X_train_s = _transform_3d(X_train, scaler)
        X_valid_s = _transform_3d(X_valid, scaler) if X_valid is not None else None
        X_test_s  = _transform_3d(X_test,  scaler) if X_test  is not None else None

    elif X_train.ndim == 2:
        n, tot = X_train.shape
        if window_size is not None:
            # Reescalar por feature: reconstruyo 3D -> escalo -> vuelvo a 2D
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size

            def to3d(X2d):
                return X2d.reshape(X2d.shape[0], window_size, F)

            Xtr3 = to3d(X_train)
            if not fitted:
                scaler = _fit_on_3d(Xtr3, scaler)

            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)

            if X_valid is not None:
                Xva3 = to3d(X_valid)
                X_valid_s = _transform_3d(Xva3, scaler).reshape(X_valid.shape[0], tot)
            else:
                X_valid_s = None

            if X_test is not None:
                Xte3 = to3d(X_test)
                X_test_s = _transform_3d(Xte3, scaler).reshape(X_test.shape[0], tot)
            else:
                X_test_s = None
        else:
            # Escalado columna a columna (no reconstruye 3D)
            if not fitted:
                scaler.fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_valid_s = scaler.transform(X_valid) if X_valid is not None else None
            X_test_s  = scaler.transform(X_test)  if X_test  is not None else None
    else:
        raise ValueError("X_train debe ser 2D o 3D.")

    # ---------------------------------------------------------
    # 3) Guardar escalador (el que efectivamente se usó)
    # ---------------------------------------------------------
    joblib.dump(scaler, scaler_path)
    if verbose:
        print(f"✅ Scaler usado/actualizado guardado en: {scaler_path}")
        print("Shapes escaladas:",
              "X_train", X_train_s.shape,
              "| X_valid", None if X_valid is None else X_valid_s.shape,
              "| X_test",  None if X_test  is None  else X_test_s.shape)

    return X_train_s, X_valid_s, X_test_s, scaler


### 3.1. Definir rutas de ventanas escaladas:

Definimos las rutas

In [64]:
# Ruta base donde guardarás los folds
ruta_k_scaler = f'{drive_path}/5_transformer_90_model/5_2_k_scaler'
os.makedirs(ruta_k_scaler, exist_ok=True)

In [65]:
def xy_paths_for_fold(k: int):
    # Crear carpeta del fold
    fold_path = os.path.join(ruta_k_scaler, f"fold_{k}")
    os.makedirs(fold_path, exist_ok=True)
    base = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}'
    return {
        "X_train_sc": f"{base}/X_train_sc_{k}.npz",
        "y_train_sc": f"{base}/y_train_sc_{k}.npz",
        "X_valid_sc": f"{base}/X_valid_sc_{k}.npz",
        "y_valid_sc": f"{base}/y_valid_sc_{k}.npz",
        "X_test_sc":  f"{base}/X_test_sc_{k}.npz",
        "y_test_sc":  f"{base}/y_test_sc_{k}.npz",
    }

In [66]:
K = 5

rutas_scaler = {
    k: xy_paths_for_fold(k)
    for k in range(1, K + 1)
}

In [67]:
#rutas_scaler

### 3.2. Aplicación de escalamiento

#### 3.2.1. Función para guardar o cargar ventanas escaladas:

In [68]:
def save_or_load_scaled_data(path_train, path_valid, path_test, horizonte, X_train, X_valid, X_test, y_train, y_valid, y_test):

  paths = {
    "train": path_train,
    "valid": path_valid,
    "test":  path_test
    }

  scaler_path = f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl"

  # Verificar existencia conjunta
  if all(os.path.exists(p) for p in paths.values()):
      print("Los tres archivos existen → Cargando ventanas escaladas")

      data_train_s = np.load(path_train)
      X_train_s, y_train = data_train_s["X"], data_train_s["y"]

      data_valid_s = np.load(path_valid)
      X_valid_s, y_valid = data_valid_s["X"], data_valid_s["y"]

      data_test_s = np.load(path_test)
      X_test_s, y_test = data_test_s["X"], data_test_s["y"]

      print("Carga completa.")


  else:
      print("Alguno de los archivos no existe → Generando ventanas escaladas")

      if os.path.exists(scaler_path):
        print(f"Scaler existente → cargando global_scaler.pkl")
        scaler = joblib.load(scaler_path)
      else:
        print(f"Scaler no existe → creando global_scaler.pkl")

      X_train_s, X_valid_s, X_test_s, scaler = scale_and_save(
        X_train,
        X_valid,
        X_test,
        scaler_type="standard",
        scaler_path=scaler_path,
        window_size=90,        # ← IMPORTANTE para su caso
        verbose=True,
        reuse_if_exists=True   # ← usa scaler guardado si existe
    )


      np.savez_compressed(path_train, X=X_train_s, y=y_train)
      np.savez_compressed(path_valid, X=X_valid_s, y=y_valid)
      np.savez_compressed(path_test,  X=X_test_s,  y=y_test)

      print("Guardado completo.")

  return X_train_s, X_valid_s, X_test_s

#### 3.2.2. Aplicación de escalamiento

In [69]:
global_scaler_path = f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl"

In [70]:
import gc, os, ctypes
import numpy as np
import joblib

# Opcional: si definiste antes ram_usage(), la podés usar aquí.
# def ram_usage(): ...

def scale_apply_from_disk(fold: int, window_size: int = None, reuse_if_exists: bool = True, show_ram: bool = False):
    print(f"\n==============================")
    print(f"▶ Fold {fold} - INICIO")
    print(f"==============================")

    if show_ram and 'ram_usage' in globals():
        ram_usage()

    # ---------- 1) Asegurar directorio del scaler ----------
    scaler_dir = os.path.dirname(global_scaler_path)
    if scaler_dir and not os.path.exists(scaler_dir):
        os.makedirs(scaler_dir, exist_ok=True)

    # ---------- 2) Cargar o crear scaler ----------
    fitted = False
    if reuse_if_exists and os.path.exists(global_scaler_path):
        scaler = joblib.load(global_scaler_path)
        fitted = True
        print(f"Usando scaler existente: {global_scaler_path}")
    else:
        scaler = _choose_scaler("standard")
        print("Scaler no encontrado o reuse_if_exists=False → se creará uno nuevo.")

    train_path = rutas_ventanas[fold]["X_train"]
    valid_path = rutas_ventanas[fold]["X_valid"]
    test_path  = rutas_ventanas[fold]["X_test"]

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"No se encontró el archivo de train para fold {fold}: {train_path}")

    # Para reusar info de forma (n, W, F) en valid/test
    is_3d = None
    F = None
    tot = None

    # ---------- 3) PROCESAR TRAIN ----------
    print(f"[TRAIN] Cargando ventanas desde: {train_path}")
    train_npz = np.load(train_path)
    X_train = train_npz["X"].astype(np.float32)
    y_train = train_npz["y"]

    print(f"[TRAIN] Shape X_train: {X_train.shape}, y_train: {y_train.shape}")
    if show_ram and 'ram_usage' in globals():
        ram_usage()

    if X_train.ndim == 3:
        is_3d = True
        _, W, F = X_train.shape
        print(f"[TRAIN] Datos 3D → (n, W={W}, F={F})")

        if not fitted:
            print("[TRAIN] Ajustando scaler con _fit_on_3d(X_train, scaler)...")
            scaler = _fit_on_3d(X_train, scaler)

        print("[TRAIN] Escalando con _transform_3d...")
        X_train_s = _transform_3d(X_train, scaler)

    elif X_train.ndim == 2:
        is_3d = False
        n, tot = X_train.shape
        print(f"[TRAIN] Datos 2D → (n={n}, tot={tot})")

        if window_size is not None:
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size
            print(f"[TRAIN] window_size={window_size}, F={F} features por paso.")

            def to3d(X2):
                return X2.reshape(X2.shape[0], window_size, F)

            print("[TRAIN] Reshape a 3D para ajustar/transformar...")
            Xtr3 = to3d(X_train)

            if not fitted:
                print("[TRAIN] Ajustando scaler con _fit_on_3d(Xtr3, scaler)...")
                scaler = _fit_on_3d(Xtr3, scaler)

            print("[TRAIN] Escalando con _transform_3d y volviendo a 2D...")
            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)
            del Xtr3
        else:
            if not fitted:
                print("[TRAIN] Ajustando scaler con scaler.fit(X_train)...")
                scaler.fit(X_train)
            print("[TRAIN] Escalando con scaler.transform(X_train)...")
            X_train_s = scaler.transform(X_train)
    else:
        del X_train, y_train, train_npz
        raise ValueError("X_train debe ser 2D o 3D.")

    # Guardar train escalado y liberar memoria
    print(f"[TRAIN] Guardando escalado en: {rutas_scaler[fold]['X_train_sc']}")
    np.savez_compressed(rutas_scaler[fold]["X_train_sc"], X=X_train_s, y=y_train)

    print("[TRAIN] Eliminando X_train_s, X_train, y_train, train_npz de memoria...")
    del X_train_s, X_train, y_train, train_npz
    gc.collect()
    if show_ram and 'ram_usage' in globals():
        ram_usage()

    # ---------- 4) Función auxiliar para VALID / TEST ----------
    def procesar_split(path_in, path_out, nombre):
        if not os.path.exists(path_in):
            print(f"[{nombre.upper()}] Archivo no encontrado para fold {fold}: {path_in}")
            return

        print(f"[{nombre.upper()}] Cargando ventanas desde: {path_in}")
        npz = np.load(path_in)
        X = npz["X"].astype(np.float32)
        y = npz["y"]
        print(f"[{nombre.upper()}] Shape X: {X.shape}, y: {y.shape}")

        if is_3d:  # ya sabemos que train era 3D
            print(f"[{nombre.upper()}] Escalando con _transform_3d...")
            X_s = _transform_3d(X, scaler)
        else:
            if window_size is not None:
                n = X.shape[0]

                def to3d(X2):
                    return X2.reshape(X2.shape[0], window_size, F)

                print(f"[{nombre.upper()}] Reshape a 3D para escalar y volver a 2D...")
                X3 = to3d(X)
                X_s = _transform_3d(X3, scaler).reshape(n, tot)
                del X3
            else:
                print(f"[{nombre.upper()}] Escalando con scaler.transform(X)...")
                X_s = scaler.transform(X)

        print(f"[{nombre.upper()}] Guardando escalado en: {path_out}")
        np.savez_compressed(path_out, X=X_s, y=y)

        print(f"[{nombre.upper()}] Eliminando X_s, X, y, npz de memoria...")
        del X_s, X, y, npz
        gc.collect()
        if show_ram and 'ram_usage' in globals():
            ram_usage()

    # ---------- 5) PROCESAR VALID Y TEST UNO POR VEZ ----------
    procesar_split(
        valid_path,
        rutas_scaler[fold]["X_valid_sc"],
        "validación"
    )

    procesar_split(
        test_path,
        rutas_scaler[fold]["X_test_sc"],
        "test"
    )

    # ---------- 6) Guardar scaler y liberar ----------
    joblib.dump(scaler, global_scaler_path)
    print(f"Scaler guardado/actualizado en: {global_scaler_path}")

    print("Liberando scaler de memoria...")
    del scaler
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

    print("Archivos escalados guardados:")
    print("   -", rutas_scaler[fold]["X_train_sc"])
    print("   -", rutas_scaler[fold]["X_valid_sc"])
    print("   -", rutas_scaler[fold]["X_test_sc"])
    print(f"Fold {fold} escalado, guardado y memoria liberada.\n")


In [71]:
for k in k_folds:
    scale_apply_from_disk(
        fold = k,
        window_size = window_size,
        reuse_if_exists = True,
        show_ram = True      # si querés ver el uso de memoria en cada paso
    )


▶ Fold 1 - INICIO
RAM total:      12.67 GB
RAM usada:      3.11 GB
RAM disponible: 9.24 GB
Scaler no encontrado o reuse_if_exists=False → se creará uno nuevo.
[TRAIN] Cargando ventanas desde: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_1_k_windows/fold_1/X_train_1.npz
[TRAIN] Shape X_train: (124279, 1080), y_train: (124279,)
RAM total:      12.67 GB
RAM usada:      3.62 GB
RAM disponible: 8.74 GB
[TRAIN] Datos 2D → (n=124279, tot=1080)
[TRAIN] window_size=90, F=12 features por paso.
[TRAIN] Reshape a 3D para ajustar/transformar...
[TRAIN] Ajustando scaler con _fit_on_3d(Xtr3, scaler)...
[TRAIN] Escalando con _transform_3d y volviendo a 2D...
[TRAIN] Guardando escalado en: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_2_k_scaler/fold_1/X_train_sc_1.npz
[TRAIN] Eliminando X_train_s, X_train, y_train, train_npz de memoria...
RAM total:      12.67 GB
RAM usada:      3.12 GB
RAM disponible: 9.23 GB
[VALIDACIÓN] Cargando ventanas desde: /content/drive/MyDri

In [72]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      2.90 GB
RAM disponible: 9.46 GB
